# Text Data

Python version of the R Markdown notebook on visualizing text data (word frequencies,
word clouds, ranked bar charts, lollipop charts, treemaps, co-occurrence networks,
and topic models).

**Translation notes (read once, applies throughout):**
- R's `tidytext::unnest_tokens()` is approximated here with a regex tokenizer. It matches
  the original closely enough that URLs still split into `https` / `t.co` tokens (so the
  same `badWords` trick works), and contractions like `don't` stay as one token.
- R's `tidytext::stop_words` bundles three lexicons (~1,150 unique words). Here we use
  scikit-learn's built-in `ENGLISH_STOP_WORDS` (318 words) — smaller, but it needs no
  download and works offline. If you want something closer in size to tidytext's list,
  swap in `nltk.corpus.stopwords` (requires `nltk.download('stopwords')`, i.e. internet
  access) or combine both sets.
- Counts, top words, and topic contents will differ somewhat from the R output because
  of the two points above — not because the logic is wrong.


Text can also be visualized, but unlike numeric variables, text usually needs some
extra preparation before plotting. Words appear in different forms, some words are very
common but not very informative, and some meaningful ideas are expressed as phrases
rather than as single words.

In this notebook, you will learn a simple workflow for visualizing text data. We will
begin with **word frequencies**, and then compare different graphical options:

- word clouds
- ranked bar charts
- lollipop charts
- treemaps
- topic models
- co-occurrence networks

The goal is not only to produce attractive plots, but also to understand what each plot
is good for.

## 1. Get the text data

Let us begin with a data frame containing some old tweets from D. Trump:

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from scipy.sparse import coo_matrix
import networkx as nx
from wordcloud import WordCloud
import squarify

pd.set_option("display.max_colwidth", 120)

link1 = "https://github.com/MagallanesAtAlacip/datafiles/"
link2 = "raw/main/text/trumps.csv"
trumpLink = link1 + link2
allTweets = pd.read_csv(trumpLink)

The data has these columns:

In [ ]:
list(allTweets.columns)

The column with the tweet text contains entries like these:

In [ ]:
allTweets['text'].head(2)

## 2. Select the observations of interest

This data frame has several columns that allow subsetting. In this example, I will keep
only tweets that are **not retweets**:

In [ ]:
SomeTrumpTweets = allTweets[allTweets['is_retweet'] == False]
SomeTrumpTweets = SomeTrumpTweets[['created_at', 'text']].reset_index(drop=True)
SomeTrumpTweets.head(3)

At this point, we still have complete text messages. To visualize text, we first
need to break each message into smaller units.

## 3. Turn text into plottable words

This process starts by what is called **tokenization**: splitting phrases into
individual words.

In [ ]:
TOKEN_RE = re.compile(r"[a-z]+(?:[.'][a-z]+)*")

def tokenize_words(text):
    "Approximate equivalent of unnest_tokens(token = 'words')."
    return TOKEN_RE.findall(str(text).lower())

rows = []
for text in SomeTrumpTweets['text']:
    for word in tokenize_words(text):
        rows.append(word)
pd.DataFrame({'EachWord': rows}).head()

Notice that we named a new column `EachWord`, and in fact it has just one word from
the tweet. The tweet message has been split or expanded into multiple rows, one row per
word. It is useful to keep track of which tweet each word came from, like a long format
ID. Currently, `created_at` might work, but a simpler solution is to create a sequential
ID:

In [ ]:
SomeTrumpTweets = SomeTrumpTweets.assign(tweet_id=range(1, len(SomeTrumpTweets) + 1))

rows = []
for _, r in SomeTrumpTweets.iterrows():
    for word in tokenize_words(r['text']):
        rows.append({'created_at': r['created_at'], 'tweet_id': r['tweet_id'], 'EachWord': word})
pd.DataFrame(rows).head()

Let's save that object:

In [ ]:
WordsIn_SomeTrumpTweets = pd.DataFrame(rows)[['created_at', 'tweet_id', 'EachWord']]
WordsIn_SomeTrumpTweets.head()

Notice that the date is repeated for every word coming from the same tweet.

How many word-tokens do we have now?

In [ ]:
len(WordsIn_SomeTrumpTweets)

This number includes repeated words, which is useful because frequency is often our
first summary of text. We still have some pending pre-processing.

### 3.1. Remove common words: stopwords

Many very frequent words carry little substantive meaning. Words such as *the*, *and*,
*of*, or *to* are useful for grammar, but not always useful for analysis. These are
called **stopwords**.

In [ ]:
stop_words = pd.DataFrame({'word': sorted(ENGLISH_STOP_WORDS)})
stop_words.head()

Now let us remove them:

In [ ]:
WordsIn_SomeTrumpTweets = WordsIn_SomeTrumpTweets[
    ~WordsIn_SomeTrumpTweets['EachWord'].isin(stop_words['word'])
]
len(WordsIn_SomeTrumpTweets)

After removing stopwords, the number of rows decreases because many very common
words are filtered out.

This does **not** mean the remaining words are automatically important. It only means we
are removing words that are usually too general to help interpretation.

## 3.2. Getting rid of other unneeded words

Take a look at the most repeated words (tokens):

In [ ]:
WordsIn_SomeTrumpTweets['EachWord'].value_counts().sort_values().tail(10)

Then we can safely drop some of those:

In [ ]:
badWords = ["https", "t.co"]
WordsIn_SomeTrumpTweets_clean = WordsIn_SomeTrumpTweets[
    ~WordsIn_SomeTrumpTweets['EachWord'].isin(badWords)
]

Now, let's keep the ones that appeared more than 4 times:

In [ ]:
moreThan_4 = (
    WordsIn_SomeTrumpTweets_clean['EachWord']
    .value_counts()
    .rename_axis('EachWord')
    .reset_index(name='Counts')
    .sort_values('Counts', ascending=False)
)
moreThan_4 = moreThan_4[moreThan_4['Counts'] > 4]
moreThan_4.head(10)

These are the words that appear most often after basic cleaning.

Before moving to more decorative plots, it is useful to begin with the clearest graph possible.

## 4. Options for plots

### 4.1. Ranked bar chart

A ranked bar chart is usually the best option when the goal is **comparison**.

In [ ]:
top_words = moreThan_4.head(15).sort_values('Counts')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_words['EachWord'], top_words['Counts'], color="steelblue")
ax.set_xlabel("Count")
ax.set_title("Most frequent words")
plt.tight_layout()
plt.show()

This plot is very useful because it allows precise comparison. You can clearly see
which words are most frequent, their ranking, and the size of the differences between
them. For comparing frequencies, bar charts are usually better than word clouds.

### 4.2. Lollipop chart

A lollipop chart shows the same information as a bar chart, but with a lighter
appearance.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hlines(y=top_words['EachWord'], xmin=0, xmax=top_words['Counts'], color="grey")
ax.plot(top_words['Counts'], top_words['EachWord'], "o", color="steelblue", markersize=8)
ax.set_xlabel("Count")
ax.set_title("Most frequent words: lollipop chart")
plt.tight_layout()
plt.show()

This chart still prioritizes ranking and comparison, but it uses less visual weight
than bars.

### 4.3. Word cloud

Now let us build the word cloud.

In [ ]:
freqs = dict(zip(moreThan_4['EachWord'], moreThan_4['Counts']))

wc = WordCloud(width=1000, height=700, background_color="white",
               colormap="Reds", prefer_horizontal=1.0, relative_scaling=0.5)
wc = wc.generate_from_frequencies(freqs)

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word cloud of frequent terms")
plt.show()

Word clouds are often visually attractive, but they are not ideal for accurate
comparison. They are better for quick engagement and general impression than for careful
measurement.

### 4.4. Treemap

A treemap is another way to represent frequencies, this time using **area** of
rectangles.

Install `squarify` if needed:

```
# pip install squarify
```

In [ ]:
top20 = moreThan_4.head(20)
colors = plt.cm.Blues([0.3 + 0.6 * (c / top20['Counts'].max()) for c in top20['Counts']])

fig, ax = plt.subplots(figsize=(10, 7))
squarify.plot(sizes=top20['Counts'], label=top20['EachWord'], color=colors,
              text_kwargs={'color': 'white', 'fontsize': 10}, ax=ax)
ax.axis("off")
ax.set_title("Word frequencies as a treemap")
plt.show()

Treemaps are compact and visually appealing, but they are not as precise as bars
for comparing values.

## 5. A second example: reading from a TXT file

Not all text data comes from tweets. Sometimes you simply have a text file.

We can read it like this:

In [ ]:
LinkText = "https://github.com/MagallanesAtAlacip/datafiles/raw/main/text/sometext.txt"
otherText_raw = pd.read_csv(LinkText, sep="\t", header=None, names=["V1"],
                             skip_blank_lines=True, engine="python")
otherText = otherText_raw[otherText_raw['V1'].astype(str).str.strip() != ""].reset_index(drop=True)
otherText.head()

Notice that a new row is created whenever a new paragraph or line break is found.

Now let us tokenize the text and remove stopwords and the above discovered `badWords`:

In [ ]:
rows = []
for i, line in enumerate(otherText['V1']):
    for word in tokenize_words(line):
        rows.append({'line_id': i, 'EachWord': word})
otherText_words = pd.DataFrame(rows)
otherText_words = otherText_words[~otherText_words['EachWord'].isin(stop_words['word'])]
otherText_words = otherText_words[~otherText_words['EachWord'].isin(badWords)]
otherText_words.head(20)

Now compute frequencies:

In [ ]:
txt_counts = (
    otherText_words['EachWord']
    .value_counts()
    .rename_axis('EachWord')
    .reset_index(name='Counts')
    .sort_values('Counts', ascending=False)
)
txt_counts = txt_counts[txt_counts['Counts'] > 4]
txt_counts.head(10)

## 6. Other plots beyond raw counts

Once you have word counts or frequencies, you can prepare your word cloud:

In [ ]:
freqs_txt = dict(zip(txt_counts['EachWord'], txt_counts['Counts']))
wc2 = WordCloud(width=1000, height=700, background_color="white",
                colormap="Reds", prefer_horizontal=1.0).generate_from_frequencies(freqs_txt)

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(wc2, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word cloud from the TXT file")
plt.show()

And your ranked bars:

In [ ]:
txt_top15 = txt_counts.head(15).sort_values('Counts')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(txt_top15['EachWord'], txt_top15['Counts'], color="steelblue")
ax.set_xlabel("Count")
ax.set_title("Most frequent words in the text")
plt.tight_layout()
plt.show()

You may also use proportions instead of raw values as labels (counts still are represented on the x-axis):

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(txt_top15['EachWord'], txt_top15['Counts'], color="steelblue")
pct = 100 * txt_top15['Counts'] / txt_top15['Counts'].sum()
for y, (count, p) in enumerate(zip(txt_top15['Counts'], pct)):
    ax.text(count + 0.3, y, f"{p:.1f}%", va="center")
ax.set_xlabel("Count")
ax.set_title("Most frequent words in the text")
plt.tight_layout()
plt.show()

However, other strategies beyond plotting frequencies are possible.

### 6.1 Co-occurrence networks

So far, we have treated words as isolated units. But language also has structure: some
words tend to appear together.

A simple way to explore this is through **bigrams**, that is, pairs of words that appear
next to each other. We tokenize again, but this time request consecutive pairs:

In [ ]:
bigram_rows = []
for _, r in SomeTrumpTweets.iterrows():
    toks = tokenize_words(r['text'])
    for w1, w2 in zip(toks[:-1], toks[1:]):
        bigram_rows.append({'word1': w1, 'word2': w2})
bigrams_inColumns = pd.DataFrame(bigram_rows)
bigrams_inColumns.head()

Unlike `unnest_tokens(token = "ngrams", n = 2)` followed by
`separate_wider_delim()`, the regex tokenizer above lets us build word pairs directly, so
there is no separate "splitting" step needed here.

Some cleaning and sorted counts:

In [ ]:
mask = (
    ~bigrams_inColumns['word1'].isin(stop_words['word']) &
    ~bigrams_inColumns['word2'].isin(stop_words['word']) &
    ~bigrams_inColumns['word1'].isin(badWords) &
    ~bigrams_inColumns['word2'].isin(badWords)
)
bigram_counts_clean = (
    bigrams_inColumns[mask]
    .groupby(['word1', 'word2'])
    .size()
    .reset_index(name='n')
    .sort_values('n', ascending=False)
)
bigram_counts_clean = bigram_counts_clean[bigram_counts_clean['n'] >= 5]
bigram_counts_clean.head(10)

#### 6.1.1 The creation of networks

As we have pairs, we can think of each word as a network node. If words are nodes, a
link between them means they co-occur. Notice this network model allows that a word may
be connected to more than one other word.

We use `networkx` here:

In [ ]:
bigram_graph = nx.from_pandas_edgelist(bigram_counts_clean, 'word1', 'word2', edge_attr='n')
print(bigram_graph)

Above you see new info: `bigram_graph` is an **undirected** graph — we do not care
which word came before the other, only that they co-occurred — with a certain number of
nodes and edges.

The graph also has attributes: node names, and an edge attribute `n` coming from our
counts column in `bigram_counts_clean`.

In [ ]:
list(bigram_graph.nodes())

In [ ]:
[d['n'] for _, _, d in bigram_graph.edges(data=True)]

You can easily produce a bigram plot like this:

In [ ]:
pos = nx.spring_layout(bigram_graph, seed=42, k=0.6)  # spring_layout = Fruchterman-Reingold, same family as ggraph's "fr"

fig, ax = plt.subplots(figsize=(9, 7))
nx.draw_networkx_edges(bigram_graph, pos, ax=ax)
nx.draw_networkx_nodes(bigram_graph, pos, node_size=120, ax=ax)
ax.axis("off")
plt.show()

The hardest part of drawing networks is deciding where to place nodes. The
Fruchterman-Reingold layout (`spring_layout` in networkx) helps by simulating attraction
between connected nodes and repulsion among all nodes. This usually spreads the graph
clearly while bringing related nodes together, making clusters and central nodes easier
to see.

We can improve this by adding names to the nodes:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
nx.draw_networkx_edges(bigram_graph, pos, ax=ax)
nx.draw_networkx_nodes(bigram_graph, pos, node_size=120, ax=ax)
nx.draw_networkx_labels(bigram_graph, pos, font_size=8, ax=ax)
ax.axis("off")
plt.show()

We know that those edges have the counts attribute (`n`). We can use the line width channel to present the weight:

In [ ]:
weights = [bigram_graph[u][v]['n'] for u, v in bigram_graph.edges()]

fig, ax = plt.subplots(figsize=(9, 7))
nx.draw_networkx_edges(bigram_graph, pos, width=weights, ax=ax)
nx.draw_networkx_nodes(bigram_graph, pos, node_size=120, ax=ax)
nx.draw_networkx_labels(bigram_graph, pos, font_size=8, ax=ax)
ax.axis("off")
plt.show()

Let's add some details for improvement:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
nx.draw_networkx_edges(bigram_graph, pos, width=weights, alpha=0.4, edge_color="blue", ax=ax)
nx.draw_networkx_nodes(bigram_graph, pos, node_size=140, node_color="black", ax=ax)
nx.draw_networkx_labels(bigram_graph, pos, font_size=9, ax=ax)
ax.axis("off")
plt.show()

# Note: ggraph's geom_node_text(repel = TRUE) nudges overlapping labels apart.
# networkx has no built-in equivalent; the optional "adjustText" package
# (pip install adjustText) can do this on top of the labels drawn above.

This plot does not focus only on word frequency. Instead, it reveals
**relationships between words**. That is often more meaningful than treating words
separately.

### 6.2 Topic models

So far, we have focused on **individual words**: how often they appear, how they relate
to other words, or how they can be visualized. But sometimes we are interested in
something broader: the **main themes** discussed across many documents.

This is where **topic models** become useful.

A topic model is a statistical method that looks for groups of words that tend to appear
together repeatedly across documents. When certain words frequently co-occur, the model
interprets that pattern as a possible **topic**.

For example, if many tweets contain words such as:

- jobs
- economy
- workers
- manufacturing

the model may detect an **economic topic**.

If another group repeatedly contains:

- china
- trade
- tariffs
- imports

the model may detect a **trade policy topic**.

The important point is that the model does **not** know these labels beforehand. It only
detects patterns of word co-occurrence. Humans then inspect the results and assign
meaningful interpretations.

In short, instead of asking:

> Which words are most frequent?

topic models ask:

> Which words tend to appear together across documents, suggesting an underlying theme?

Here we need counting again, but **not per corpus** — per document:

In [ ]:
countWordsPerTweets = (
    WordsIn_SomeTrumpTweets_clean
    .groupby(['tweet_id', 'EachWord'])
    .size()
    .reset_index(name='n')
)
countWordsPerTweets.head()

The object `countWordsPerTweets` is still a regular data frame in **long format**.
Each row contains one tweet (`tweet_id`), one word (`EachWord`), and how many times that
word appeared in that tweet (`n`).

This format is excellent for inspection and manipulation, but many text-mining models
require a different structure: a **Document-Term Matrix** (DTM).

In [ ]:
doc_codes = countWordsPerTweets['tweet_id'].astype('category')
term_codes = countWordsPerTweets['EachWord'].astype('category')

tweet_DTM = coo_matrix(
    (countWordsPerTweets['n'], (doc_codes.cat.codes, term_codes.cat.codes))
).tocsr()

doc_ids = doc_codes.cat.categories
terms = term_codes.cat.categories

`tweet_DTM` is a matrix created from the text counts where rows are documents
(tweets), columns are words, and cells contain how many times each word appears in each
tweet.

We have:

In [ ]:
n_docs, n_terms = tweet_DTM.shape
nnz = tweet_DTM.nnz
total_cells = n_docs * n_terms
sparsity = 100 * (total_cells - nnz) / total_cells
longest_term = max(len(t) for t in terms)

print(f"<<DocumentTermMatrix ({n_docs} documents, {n_terms} terms)>>")
print(f"Non-/sparse entries: {nnz}/{total_cells - nnz}")
print(f"Sparsity           : {sparsity:.0f}%")
print(f"Maximal term length: {longest_term}")
print("Weighting          : term frequency (tf)")

The object is a **Document-Term Matrix**. Each row represents one tweet, each
column represents one word, and each cell records how many times that word appears in
that tweet. The matrix is highly sparse, which is normal in text data — especially with
short documents such as tweets — because any single tweet uses only a small fraction of
all possible words.

We can check a portion of the DTM:

In [ ]:
sub = tweet_DTM[0:5, 0:10].toarray()
sub_df = pd.DataFrame(sub, index=doc_ids[0:5], columns=terms[0:10])
sub_nnz = (sub != 0).sum()
sub_total = sub.size
print(f"{sub_nnz} cells contain values, {sub_total - sub_nnz} cells are zero "
      f"({100*(sub_total - sub_nnz)/sub_total:.0f}% sparsity)")
sub_df

A `0` means the word does not appear in the tweet (a tweet is a row).

The next step is to estimate a **topic model** using scikit-learn's
`LatentDirichletAllocation`, the Python equivalent of R's `topicmodels::LDA()`. LDA
stands for **Latent Dirichlet Allocation**, one of the most common methods for
discovering hidden themes in a collection of documents.

In the code below, `n_components=3` tells the model to search for **three topics**, so
the tweets will be represented as mixtures of three underlying themes. `random_state`
plays the same role as R's `control = list(seed = 1234)`: it makes the run reproducible.
Note that scikit-learn's variational-Bayes solver and R's `topicmodels` VEM solver are
different implementations of the same idea — expect the topics to be broadly comparable,
not numerically identical.

In [ ]:
lda_model = LatentDirichletAllocation(n_components=3, random_state=1234,
                                         learning_method="batch", max_iter=100)
lda_model.fit(tweet_DTM)
lda_model

We can extract important information from `lda_model`: the probability of each word within each topic (`beta`):

In [ ]:
beta_matrix = lda_model.components_ / lda_model.components_.sum(axis=1, keepdims=True)
topics_terms_betas = (
    pd.DataFrame(beta_matrix, columns=terms)
    .reset_index()
    .rename(columns={'index': 'topic'})
    .melt(id_vars='topic', var_name='term', value_name='beta')
)
topics_terms_betas['topic'] = topics_terms_betas['topic'] + 1  # 1-indexed, like R
topics_terms_betas

The object `topics_terms_betas` shows the probability of each word within each
topic. In other words, beta measures how strongly a word is associated with a topic.
Very small values mean that the word is almost absent from that topic, while larger
values indicate that the word is more characteristic of it. To interpret the model, we
usually focus on the words with the highest beta values in each topic.

Let us extract the most important terms in each topic:

In [ ]:
topics_terms_betas = (
    topics_terms_betas
    .sort_values(['topic', 'beta'], ascending=[True, False])
    .groupby('topic')
    .head(8)
)

To plot them nicely (equivalent of `reorder_within()` + `scale_x_reordered()` + `facet_wrap()`), we sort within each topic and give each subplot its own y-axis:

In [ ]:
topic_ids = sorted(topics_terms_betas['topic'].unique())
fig, axes = plt.subplots(1, len(topic_ids), figsize=(14, 5), sharex=False)

for ax, t in zip(axes, topic_ids):
    sub = topics_terms_betas[topics_terms_betas['topic'] == t].sort_values('beta')
    ax.barh(sub['term'], sub['beta'], color="steelblue")
    ax.set_title(f"topic: {t}")
    ax.set_xlabel("Beta")

fig.suptitle("Top terms by topic")
plt.tight_layout()
plt.show()

This is where we end. I hope you liked the journey!